![Logo del Proyecto](https://www.utdt.edu/Images/prensa/utdt-color-baja.png)

<h1 align="center"> Tesis - Master in Management + Analytics </h1>
<h2 align="center"> IA como Herramienta para el Control de Calidad de Imágenes<br> en el Diagnóstico de Retinopatía Diabética </h2>

**Fecha:** Octubre 2024<br>
**Autora:** Gabriela Moran<br>
**Tutor:** [Santiago Cisco](https://ar.linkedin.com/in/mariocisco)<br>

In [1]:
from pathlib import Path
import os
import sys
import pandas as pd
import numpy as np
import configparser

In [2]:
MiM_path = Path(os.getcwd()).parent
DIR = sys.path.append(MiM_path/'data')
BASE = Path(MiM_path/'data')

# Cantidad de imágenes de mala calidad en cada dataset

In [54]:
global_labels = configparser.ConfigParser()
global_labels.optionxform = str
global_labels.read(BASE/'labels/global_labels.ini')

['c:\\Users\\gabim\\MiM\\data\\labels\\global_labels.ini']

In [55]:
global_bin = configparser.ConfigParser()
global_bin.read(BASE/'splits/global_binaria.ini')

['c:\\Users\\gabim\\MiM\\data\\splits\\global_binaria.ini']

In [56]:
all_data = pd.DataFrame(global_labels['label'].items(), columns=['filename','label'])

In [59]:
all_data.label = all_data.label.astype(int)

In [60]:
train = pd.DataFrame({'filename': global_bin['split'].get('training').split(sep=','), 'partition':'train'})
val = pd.DataFrame({'filename': global_bin['split'].get('validation').split(sep=','), 'partition':'val'})
test = pd.DataFrame({'filename': global_bin['split'].get('test').split(sep=','),'partition':'test'})

In [61]:
all_partitions = pd.concat([train,val,test],axis=0,ignore_index=True)

In [62]:
all_partitions = all_partitions.merge(all_data, how='inner',on ='filename')

In [63]:
all_partitions[['dataset','filename']] = all_partitions['filename'].str.split(pat='/',expand=True)

In [64]:
all_partitions['label'] = all_partitions.label.apply(lambda x: 'Mala' if x == 1 else 'Buena')

In [66]:
resumen = all_partitions.pivot_table(index='dataset',
                                      columns=['partition','label'],
                                      aggfunc={'partition':'count'},
                                      fill_value= 0)

In [67]:
resumen.columns = resumen.columns.droplevel()

In [68]:
resumen.columns.set_names(['',''],inplace=True)

In [69]:
resumen

test       train        \
                                                   Buena Mala  Buena  Mala   
dataset                                                                      
DDR-dataset                                         3759  346   6260   575   
Deep-Diabetic-Retinopathy-Image-Dataset-DeepDRiD-    758  842      0     0   
HRF - Quality                                          3    3      9     9   
Kaggle                                             41797  873  33841  1285   

                                                     val       
                                                   Buena Mala  
dataset                                                        
DDR-dataset                                         2503  230  
Deep-Diabetic-Retinopathy-Image-Dataset-DeepDRiD-      0    0  
HRF - Quality                                          6    6  
Kaggle                                             10680  226

In [70]:
resumen['test','Total'] = resumen['test','Buena'] + resumen['test','Mala']
resumen['train','Total'] = resumen['train','Buena'] + resumen['train','Mala']
resumen['val','Total'] = resumen['val','Buena'] + resumen['val','Mala']

In [72]:
resumen = resumen.sort_index(axis=1)
resumen = resumen.reindex(level=0,columns=['train','val','test'])

In [75]:
resumen = pd.concat([resumen, pd.DataFrame(resumen.sum(axis=0)).T.rename(index={0: 'Total'})])

In [77]:
resumen['train','Buena'] = round(resumen['train','Buena']/resumen['train','Total'],2).fillna(0)
resumen['train','Mala'] = round(resumen['train','Mala']/resumen['train','Total'],2).fillna(0)

resumen['val','Buena'] = round(resumen['val','Buena']/resumen['val','Total'],2).fillna(0)
resumen['val','Mala'] = round(resumen['val','Mala']/resumen['val','Total'],2).fillna(0)

resumen['test','Buena'] = round(resumen['test','Buena']/resumen['test','Total'],2).fillna(0)
resumen['test','Mala'] = round(resumen['test','Mala']/resumen['test','Total'],2).fillna(0)


In [78]:
resumen

train                val  \
                                                  Buena  Mala  Total Buena   
DDR-dataset                                        0.92  0.08   6835  0.92   
Deep-Diabetic-Retinopathy-Image-Dataset-DeepDRiD-  0.00  0.00      0  0.00   
HRF - Quality                                      0.50  0.50     18  0.50   
Kaggle                                             0.96  0.04  35126  0.98   
Total                                              0.96  0.04  41979  0.97   

                                                                test        \
                                                   Mala  Total Buena  Mala   
DDR-dataset                                        0.08   2733  0.92  0.08   
Deep-Diabetic-Retinopathy-Image-Dataset-DeepDRiD-  0.00      0  0.47  0.53   
HRF - Quality                                      0.50     12  0.50  0.50   
Kaggle                                             0.02  10906  0.98  0.02   
Total                                              0.03  13651  0.96  0.04   

                                                          
                                                   Total  
DDR-dataset                                         4105  
Deep-Diabetic-Retinopathy-Image-Dataset-DeepDRiD-   1600  
HRF - Quality                                          6  
Kaggle                                             42670  
Total                                              48381

# Output

In [79]:
resumen.to_excel(BASE/'reports/resumen.xlsx')